<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [8]</a>'.</span>

In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
import GWFish.modules as gw


os.environ["CUDA_VISIBLE_DEVICES"] = "3"  # use GPU 1

name="stage1_2"

waveform_model = 'IMRPhenomXPHM'
f_ref = 20.
detectors = ['ET']
network = gw.detection.Network(detector_ids = detectors, detection_SNR = (0., 8.))


## Load Model

In [ ]:
from dingo.core.posterior_models.build_model import build_model_from_kwargs
main_pm = build_model_from_kwargs(
    filename=f"/scratch/tpausch/models/{name}/model_latest.pt",
    device="cuda",
    load_training_info=False,
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data = np.loadtxt(f"/scratch/tpausch/models/{name}/history.txt")
epochs   = data[:, 0]
train    = data[:, 1]
val      = data[:, 2]


plt.figure(figsize=(8, 5))
plt.plot(epochs, train, label="Train loss")
plt.plot(epochs, val,   label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Log-likelihood")
plt.legend()
plt.title("Dingo training history")
plt.tight_layout()
plt.show()

## Simulate Injections


In [ ]:
import dingo.gw.injection as injection
from dingo.gw.noise.asd_dataset import ASDDataset

# ============================================================
# Setup
# ============================================================

num_injections = 100_000
out_dir = f"/scratch/tpausch/test_data/{name}/injections"
os.makedirs(out_dir, exist_ok=True)

np.random.seed(42)  # optional, for reproducibility

injection_generator = injection.Injection.from_posterior_model_metadata(
    main_pm.metadata
)

asd_fname = main_pm.metadata["train_settings"]["training"]["stage_0"]["asd_dataset_path"]
asd_dataset = ASDDataset(file_name=asd_fname)

injection_generator.asd = {
    k: v[0] for k, v in asd_dataset.asds.items()
}

# This is the combined intrinsic + extrinsic bilby PriorDict
# that the network was trained on
prior = injection_generator.prior
print(prior)

# ============================================================
# Draw and run injections
# ============================================================

for idx in range(num_injections):

    print(f"Running injection {idx}")

    theta = prior.sample()
    theta = {k: float(v) for k, v in theta.items()}  # cast numpy -> float

    strain_data = injection_generator.injection(theta)

    payload = {
        "theta_true": theta,
        "strain_data": strain_data,
    }

    fname = os.path.join(out_dir, f"inj_{idx:04d}.pkl")

    with open(fname, "wb") as f:
        pickle.dump(payload, f)

## Infere parameters

In [ ]:
%%capture  
#avoids printing a lot of output to the notebook
import pandas as pd
from dingo.gw.inference.gw_samplers import GWSampler




os.makedirs(f"/scratch/tpausch/test_data/{name}/results", exist_ok=True)

injection_files = sorted(os.listdir(f"/scratch/tpausch/test_data/stage1_2/injections"))

#first 100 injections
for fname in injection_files[:10000]:

    # --------------------------------------------------------
    # Load injection
    # --------------------------------------------------------

    with open(f"/scratch/tpausch/test_data/stage1_2/injections/{fname}", "rb") as f:
        payload = pickle.load(f)

    theta_true = payload["theta_true"]
    strain_data = payload["strain_data"]

    # --------------------------------------------------------
    # Run sampler
    # --------------------------------------------------------

    sampler = GWSampler(model=main_pm)

    sampler.context = strain_data

    sampler.run_sampler(
        num_samples=5_000,
        batch_size=5_000,
    )

    result = sampler.to_result()



    # --------------------------------------------------------
    # calculate network SNR
    # --------------------------------------------------------
    parameters = pd.DataFrame(
        {key: np.array([np.asarray(val).item()]) for key, val in theta_true.items()}
    )
    snr_df = gw.utilities.get_snr(parameters, network, waveform_model, f_ref)
    network_snr = float(snr_df["network"].iloc[0])  # extract as plain float

    # --------------------------------------------------------
    # Store result
    # --------------------------------------------------------

    result_payload = {
        "theta_true": theta_true,
        "result": result,
        "network_snr": network_snr,
    }

    outname = fname.replace("inj_", "result_")

    with open(f"/scratch/tpausch/test_data/{name}/results/{outname}", "wb") as f:
        pickle.dump(result_payload, f)

    print(f"Saved /scratch/tpausch/test_data/{name}/results/{outname}")

## Calculate and plot chirp mass inference error

In [ ]:
def plot_chirpmass2(name, min, max, bin_width=50):
    CM_MIN, CM_MAX = min, max
    true_cm = []
    true_mr = []
    mass_ratio_error = []
    mass_ratio_std = []
    chirp_mass_error = []
    chirp_mass_std = []
    inferred_cm = [] 

    result_files = sorted(os.listdir(f"/scratch/tpausch/test_data/{name}/results"))

    for fname in result_files:

        with open(f"/scratch/tpausch/test_data/{name}/results/{fname}", "rb") as f:
            payload = pickle.load(f)

        theta_true = payload["theta_true"]

        # --- filter on true chirp mass ---
        true_chirp_mass = theta_true["chirp_mass"]
        if not (CM_MIN <= true_chirp_mass <= CM_MAX):
            continue

        result = payload["result"]
        posterior_samples = result.samples

        cm_samples = np.array(posterior_samples["chirp_mass"])
        cm_mean = np.median(cm_samples)
        cm_std = np.std(cm_samples)

        true_mr.append(theta_true["mass_ratio"])
        true_cm.append(theta_true["chirp_mass"])
        chirp_mass_error.append(cm_mean - theta_true["chirp_mass"])
        chirp_mass_std.append(cm_std)

    true_mr = np.array(true_mr)
    true_cm = np.array(true_cm)
    cm_err = np.asarray(chirp_mass_error)

    # ------------------------------------------------------------
    # Histogram: p90 of |error| per chirp-mass bin
    # ------------------------------------------------------------
    edges = np.arange(
        np.floor(CM_MIN / bin_width) * bin_width,
        np.ceil(CM_MAX / bin_width) * bin_width + bin_width,
        bin_width,
    )
    bin_idx = np.digitize(true_cm, edges) - 1  # bin index for each event

    p90_per_bin = np.full(len(edges) - 1, np.nan)
    counts = np.zeros(len(edges) - 1, dtype=int)
    for i in range(len(edges) - 1):
        in_bin = np.abs(cm_err[bin_idx == i])
        counts[i] = len(in_bin)
        if len(in_bin) > 0:
            p90_per_bin[i] = np.percentile(in_bin, 90)

    filled = counts > 0
    plt.bar(
        edges[:-1][filled],
        p90_per_bin[filled],
        width=bin_width,
        align="edge",
        edgecolor="black",
        alpha=0.7,
    )

    # annotate number of events per bin
    for left, h, n in zip(edges[:-1][filled], p90_per_bin[filled], counts[filled]):
        plt.text(left + bin_width / 2, h, f"n={n}", ha="center", va="bottom", fontsize=8)

    plt.xlabel("True chirp mass")
    plt.ylabel("90th percentile of |inferred − true| chirp mass")
    plt.title(f"Chirp-mass inference error (p90 per {bin_width}-wide bin)")
    plt.grid(True, axis="y")
    plt.tight_layout()
    plt.axhline(50, linestyle="--", color="red", label="error = 50")
    plt.axhline(75, linestyle="--", color="red", label="error = 75")
    plt.axhline(100, linestyle="--", color="red", label="error = 100")
    plt.show()

    # ------------------------------------------------------------
    # Overall statistics (unchanged)
    # ------------------------------------------------------------
    mean_error = np.median(cm_err)
    mean_abs_error = np.median(np.abs(cm_err))
    rms_error = np.sqrt(np.median(cm_err**2))
    std_error = np.std(cm_err)
    p90_abs_error = np.percentile(np.abs(cm_err), 90)
    p95_abs_error = np.percentile(np.abs(cm_err), 95)
    p98_abs_error = np.percentile(np.abs(cm_err), 98)
    p99_abs_error = np.percentile(np.abs(cm_err), 99)

    print(len(cm_err), "samples in range", CM_MIN, "-", CM_MAX)
    print(f"Mean error (bias):      {mean_error:.3f}")
    print(f"Mean absolute error:    {mean_abs_error:.3f}")
    print(f"RMS error:              {rms_error:.3f}")
    print(f"Std of error:           {std_error:.3f}")
    print(f"90th percentile |error|: {p90_abs_error:.3f}")
    print(f"95th percentile |error|: {p95_abs_error:.3f}")
    print(f"98th percentile |error|: {p98_abs_error:.3f}")
    print(f"99th percentile |error|: {p99_abs_error:.3f}")
    return 0

In [ ]:
def plot_chirpmass3(name, min, max, bin_width=50):
    CM_MIN, CM_MAX = min, max
    true_cm = []
    true_mr = []
    mass_ratio_error = []
    mass_ratio_std = []
    chirp_mass_error = []
    chirp_mass_std = []
    inferred_cm = [] 

    result_files = sorted(os.listdir(f"/scratch/tpausch/test_data/{name}/results"))

    for fname in result_files:

        with open(f"/scratch/tpausch/test_data/{name}/results/{fname}", "rb") as f:
            payload = pickle.load(f)

        theta_true = payload["theta_true"]

        # --- filter on true chirp mass ---
        true_chirp_mass = theta_true["chirp_mass"]
        if not (CM_MIN <= true_chirp_mass <= CM_MAX):
            continue

        result = payload["result"]
        posterior_samples = result.samples

        cm_samples = np.array(posterior_samples["chirp_mass"])
        cm_mean = np.median(cm_samples)
        cm_std = np.std(cm_samples)
        inferred_cm.append(cm_mean)

        true_mr.append(theta_true["mass_ratio"])
        true_cm.append(theta_true["chirp_mass"])
        chirp_mass_error.append(cm_mean - theta_true["chirp_mass"])
        chirp_mass_std.append(cm_std)

    true_mr = np.array(true_mr)
    true_cm = np.array(true_cm)
    cm_err = np.asarray(chirp_mass_error)
    inferred_cm = np.array(inferred_cm)

    # ------------------------------------------------------------
    # Histogram: p90 of |error| per chirp-mass bin
    # ------------------------------------------------------------
    edges = np.arange(
        np.floor(inferred_cm.min() / bin_width) * bin_width,
        np.ceil(inferred_cm.max() / bin_width) * bin_width + bin_width,
        bin_width,
    )
    bin_idx = np.digitize(inferred_cm, edges) - 1
    p90_per_bin = np.full(len(edges) - 1, np.nan)
    counts = np.zeros(len(edges) - 1, dtype=int)
    for i in range(len(edges) - 1):
        in_bin = np.abs(cm_err[bin_idx == i])
        counts[i] = len(in_bin)
        if len(in_bin) > 0:
            p90_per_bin[i] = np.percentile(in_bin, 90)

    filled = counts > 0
    plt.bar(
        edges[:-1][filled],
        p90_per_bin[filled],
        width=bin_width,
        align="edge",
        edgecolor="black",
        alpha=0.7,
    )

    # annotate number of events per bin
    for left, h, n in zip(edges[:-1][filled], p90_per_bin[filled], counts[filled]):
        plt.text(left + bin_width / 2, h, f"n={n}", ha="center", va="bottom", fontsize=8)

    plt.xlabel("Infered chirp mass")
    plt.ylabel("90th percentile of |inferred − true| chirp mass")
    plt.title(f"Chirp-mass inference error (p90 per {bin_width}-wide bin of inferred mass)")
    plt.grid(True, axis="y")
    plt.tight_layout()
    plt.axhline(25, linestyle="--", color="red", label="error = 25")
    plt.axhline(50, linestyle="--", color="red", label="error = 50")
    plt.axhline(75, linestyle="--", color="red", label="error = 75")
    plt.axhline(100, linestyle="--", color="red", label="error = 100")
   
    plt.show()


    return 0

In [ ]:
def plot_chirpmass4(name, min, max, bin_width=50):
    CM_MIN, CM_MAX = min, max
    true_cm = []
    true_mr = []
    mass_ratio_error = []
    mass_ratio_std = []
    chirp_mass_error = []
    chirp_mass_std = []
    inferred_cm = [] 

    result_files = sorted(os.listdir(f"/scratch/tpausch/test_data/{name}/results"))

    for fname in result_files:

        with open(f"/scratch/tpausch/test_data/{name}/results/{fname}", "rb") as f:
            payload = pickle.load(f)

        theta_true = payload["theta_true"]

        # --- filter on true chirp mass ---
        true_chirp_mass = theta_true["chirp_mass"]
        if not (CM_MIN <= true_chirp_mass <= CM_MAX):
            continue

        result = payload["result"]
        posterior_samples = result.samples

        cm_samples = np.array(posterior_samples["chirp_mass"])
        cm_mean = np.median(cm_samples)
        cm_std = np.std(cm_samples)
        inferred_cm.append(cm_mean)

        true_mr.append(theta_true["mass_ratio"])
        true_cm.append(theta_true["chirp_mass"])
        chirp_mass_error.append(cm_mean - theta_true["chirp_mass"])
        chirp_mass_std.append(cm_std)

    true_mr = np.array(true_mr)
    true_cm = np.array(true_cm)
    cm_err = np.asarray(chirp_mass_error)
    inferred_cm = np.array(inferred_cm)

    # ------------------------------------------------------------
    # Histogram: p95 of |error| per chirp-mass bin
    # ------------------------------------------------------------
    edges = np.arange(CM_MIN, CM_MAX + bin_width, bin_width)
    bin_idx = np.digitize(inferred_cm, edges) - 1
    edges = np.arange(CM_MIN, CM_MAX, bin_width)
    edges = np.append(edges, CM_MAX)

    p90_per_bin = np.full(len(edges) - 1, np.nan)
    p95_per_bin = np.full(len(edges) - 1, np.nan)
    p98_per_bin = np.full(len(edges) - 1, np.nan)
    counts = np.zeros(len(edges) - 1, dtype=int)

    for i in range(len(edges) - 1):
        in_bin = np.abs(cm_err[bin_idx == i])
        counts[i] = len(in_bin)
        if len(in_bin) > 0:
            p90_per_bin[i], p95_per_bin[i], p98_per_bin[i] = np.percentile(in_bin, [90, 95, 98])

    filled = counts > 0
    left = edges[:-1][filled]
    p90 = p90_per_bin[filled]
    p95 = p95_per_bin[filled]
    p98 = p98_per_bin[filled]


    plt.figure(figsize=(16, 10))
    # blue: 0 -> p90, purple: p90 -> p95, red: p95 -> p98
    plt.bar(left, p90, width=bin_width, align="edge",
            color="tab:blue", edgecolor="black", alpha=0.8, label="p90")
    plt.bar(left, p95 - p90, bottom=p90, width=bin_width, align="edge",
            color="purple", edgecolor="black", alpha=0.8, label="p95")
    plt.bar(left, p98 - p95, bottom=p95, width=bin_width, align="edge",
            color="red", edgecolor="black", alpha=0.8, label="p98")

    # annotate number of events per bin (above the top of the stack)
    for l, h, n in zip(left, p98, counts[filled]):
        plt.text(l + bin_width / 2, h, f"n={n}", ha="center", va="bottom", fontsize=8)
    plt.xlabel("Infered chirp mass")
    plt.ylabel("95th percentile of |inferred − true| chirp mass")
    plt.title(f"Chirp-mass inference error (p95 per {bin_width}-wide bin of inferred mass)")
    plt.grid(True, axis="y")
    plt.tight_layout()
    plt.axhline(25, linestyle="-", color="black")
    plt.axhline(50, linestyle="-", color="black")
    plt.axhline(75, linestyle="-", color="black")
    plt.axhline(100, linestyle="-", color="black")
    plt.axhline(125, linestyle="-", color="black")
    plt.axhline(150, linestyle="-", color="black")
    plt.axhline(175, linestyle="-", color="black")
    plt.legend(title="|error| percentile", loc="upper left")
    plt.show()


    return 0

In [ ]:
plot_chirpmass4(name, 50, 1000)

In [ ]:
plot_chirpmass4(name, 100, 1000, bin_width=100)

In [ ]:
plot_chirpmass4(name, 200, 1000, bin_width=200)

In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt


def binned_percentile(x, err, edges, perr):
    """p<perr> of |err| and event count for each bin defined by edges."""
    idx = np.digitize(x, edges) - 1
    abs_err = np.abs(err)
    vals = np.full(len(edges) - 1, np.nan)
    counts = np.zeros(len(edges) - 1, dtype=int)
    for i in range(len(edges) - 1):
        in_bin = abs_err[idx == i]
        counts[i] = len(in_bin)
        if len(in_bin) > 0:
            vals[i] = np.percentile(in_bin, perr)
    return vals, counts


def edges_from_pilot(x, err, pilot_edges, lo, hi, perr, step=25, min_events=5, default_size=50):
    """
    Measure p<perr>(|err|) in the bins given by pilot_edges, then walk from lo to hi.
    The bin size at each position is the pilot perr of the pilot bin containing the
    new bin's left edge, rounded up to a multiple of `step` (minimum one step).
    """
    pilot_vals, pilot_counts = binned_percentile(x, err, pilot_edges, perr)

    edges = [lo]
    last_size = default_size
    while edges[-1] < hi:
        left = edges[-1]
        k = np.searchsorted(pilot_edges, left, side="right") - 1
        k = min(max(k, 0), len(pilot_vals) - 1)

        if pilot_counts[k] >= min_events and np.isfinite(pilot_vals[k]):
            size = step * max(1, int(np.ceil(pilot_vals[k] / step)))
        else:
            size = last_size  # too few events: keep the previous bin size
        last_size = size
        edges.append(min(left + size, hi))
    return np.array(edges)


def plot_chirpmass5(name, cm_min, cm_max, perr, pilot_width=50, step=25, min_events=5, n_it=1):
    if not (0 < perr < 100):
        raise ValueError("perr must be between 0 and 100 (e.g. 90 for the 90th percentile)")

    CM_MIN, CM_MAX = cm_min, cm_max
    inferred_cm = []
    chirp_mass_error = []

    result_files = sorted(os.listdir(f"/scratch/tpausch/test_data/{name}/results"))

    for fname in result_files:
        with open(f"/scratch/tpausch/test_data/{name}/results/{fname}", "rb") as f:
            payload = pickle.load(f)

        theta_true = payload["theta_true"]

        # --- filter on true chirp mass ---
        true_chirp_mass = theta_true["chirp_mass"]
        if not (CM_MIN <= true_chirp_mass <= CM_MAX):
            continue

        cm_samples = np.array(payload["result"].samples["chirp_mass"])
        cm_mean = np.median(cm_samples)

        inferred_cm.append(cm_mean)
        chirp_mass_error.append(cm_mean - true_chirp_mass)

    inferred_cm = np.array(inferred_cm)
    cm_err = np.asarray(chirp_mass_error)

    # ------------------------------------------------------------
    # Iterative bin-size refinement
    # ------------------------------------------------------------
    edges = np.append(np.arange(CM_MIN, CM_MAX, pilot_width), CM_MAX)  # fixed pilot bins
    for it in range(n_it):
        edges = edges_from_pilot(inferred_cm, cm_err, edges, CM_MIN, CM_MAX, perr,
                                 step=step, min_events=min_events, default_size=pilot_width)
        print(f"Iteration {it + 1}/{n_it}: {len(edges) - 1} bins")

    widths = np.diff(edges)

    # ------------------------------------------------------------
    # Bin vectors and Dingo mass ranges
    # ------------------------------------------------------------
    left_edges = edges[:-1]
    right_edges = edges[1:]
    # widths = np.diff(edges) is already defined above

    range_low = left_edges - widths     # left - width
    range_high = right_edges + widths   # right + width

    print(f"{'bin':>4} {'mass low':>15} {'mass high':>16} {'mass range':>16}")
    for i in range(len(left_edges)):
        print(f"{i:>4} {range_low[i]:>15.2f} {range_high[i]:>16.2f} {range_high[i] - range_low[i]:>16.2f}")
    
    perr_per_bin, counts = binned_percentile(inferred_cm, cm_err, edges, perr)
    filled = counts > 0

    # ------------------------------------------------------------
    # Print bin edges
    # ------------------------------------------------------------
    print("Total number of bins:", len(edges) - 1)
    print("Bin edges:", np.round(edges, 2))
    print(f"{'bin':>4} {'left':>10} {'right':>10} {'width':>10} {'n':>6} {f'p{perr} |err|':>10}")
    for i in range(len(edges) - 1):
        print(f"{i:>4} {edges[i]:>10.2f} {edges[i+1]:>10.2f} {widths[i]:>10.2f} "
              f"{counts[i]:>6} {perr_per_bin[i]:>10.2f}")

    # ------------------------------------------------------------
    # Plot: one color per bin size
    # ------------------------------------------------------------
    plt.figure(figsize=(16, 10))

    # nominal bin size (a truncated last bin is grouped with its nominal size)
    nominal = step * np.ceil(widths / step - 1e-9)
    cmap = plt.get_cmap("tab10")

    for j, size in enumerate(np.unique(nominal[filled])):
        sel = filled & (nominal == size)
        plt.bar(edges[:-1][sel], perr_per_bin[sel], width=widths[sel],
                align="edge", color=cmap(j % 10), edgecolor="black",
                alpha=0.85, label=f"bin size {size:g}")

    plt.axhline(50, linestyle="--", color="black", label="error = 50")

    for l, wi, h, n in zip(edges[:-1][filled], widths[filled],
                           perr_per_bin[filled], counts[filled]):
        plt.text(l + wi / 2, h, f"n={n}", ha="center", va="bottom", fontsize=8)

    plt.xlabel("Inferred chirp mass")
    plt.ylabel(f"{perr}th percentile of |inferred − true| chirp mass")
    plt.title(f"Chirp-mass inference error (bin size from pilot p{perr}, steps of {step})")
    plt.grid(True, axis="y")
    plt.legend(title="Bin size", loc="upper left")
    plt.tight_layout()
    plt.axhline(25, linestyle="-", color="red")
    plt.axhline(50, linestyle="-", color="red")
    plt.axhline(75, linestyle="-", color="red")
    plt.axhline(100, linestyle="-", color="red")
    plt.axhline(125, linestyle="-", color="red")
    plt.show()

    return edges, left_edges, right_edges, widths, range_low, range_high

In [ ]:
edges, left, right, widths, mr_low, mr_high = plot_chirpmass5(name, 50, 1000, 90, n_it=9)
mass_ranges = list(zip(mr_low, mr_high))

In [ ]:
edges, left, right, widths, mr_low, mr_high = plot_chirpmass5(name, 50, 1000, 90, n_it=1, bin_size=1000)
mass_ranges = list(zip(mr_low, mr_high))

In [ ]:
edges, left, right, widths, mr_low, mr_high = plot_chirpmass5(name, 50, 1000, 95, n_it=4)
mass_ranges = list(zip(mr_low, mr_high))

In [ ]:
edges, left, right, widths, mr_low, mr_high = plot_chirpmass5(name, 50, 1000, 95, n_it=4)
mass_ranges = list(zip(mr_low, mr_high))

# SNR

In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt


def binned_percentile(x, err, edges, perr):
    """p<perr> of |err| and event count for each bin defined by edges."""
    idx = np.digitize(x, edges) - 1
    abs_err = np.abs(err)
    vals = np.full(len(edges) - 1, np.nan)
    counts = np.zeros(len(edges) - 1, dtype=int)
    for i in range(len(edges) - 1):
        in_bin = abs_err[idx == i]
        counts[i] = len(in_bin)
        if len(in_bin) > 0:
            vals[i] = np.percentile(in_bin, perr)
    return vals, counts


def edges_from_pilot(x, err, pilot_edges, lo, hi, perr, step=25, min_events=5, default_size=50):
    """
    Measure p<perr>(|err|) in the bins given by pilot_edges, then walk from lo to hi.
    The bin size at each position is the pilot perr of the pilot bin containing the
    new bin's left edge, rounded up to a multiple of `step` (minimum one step).
    """
    pilot_vals, pilot_counts = binned_percentile(x, err, pilot_edges, perr)

    edges = [lo]
    last_size = default_size
    while edges[-1] < hi:
        left = edges[-1]
        k = np.searchsorted(pilot_edges, left, side="right") - 1
        k = min(max(k, 0), len(pilot_vals) - 1)

        if pilot_counts[k] >= min_events and np.isfinite(pilot_vals[k]):
            size = step * max(1, int(np.ceil(pilot_vals[k] / step)))
        else:
            size = last_size  # too few events: keep the previous bin size
        last_size = size
        edges.append(min(left + size, hi))
    return np.array(edges)


def plot_chirpmass_SNR(name, cm_min, cm_max, perr, pilot_width=50, step=25, min_events=5, n_it=1):
    if not (0 < perr < 100):
        raise ValueError("perr must be between 0 and 100 (e.g. 90 for the 90th percentile)")

    CM_MIN, CM_MAX = cm_min, cm_max
    inferred_cm = []
    chirp_mass_error = []
    snr_events=[]

    result_files = sorted(os.listdir(f"/scratch/tpausch/test_data/{name}/results"))

    for fname in result_files:
        with open(f"/scratch/tpausch/test_data/{name}/results/{fname}", "rb") as f:
            payload = pickle.load(f)

        theta_true = payload["theta_true"]
        snr_events.append(payload["network_snr"])

        # --- filter on true chirp mass ---
        true_chirp_mass = theta_true["chirp_mass"]
        if not (CM_MIN <= true_chirp_mass <= CM_MAX):
            continue

        cm_samples = np.array(payload["result"].samples["chirp_mass"])
        cm_mean = np.mean(cm_samples)

        inferred_cm.append(cm_mean)
        chirp_mass_error.append(cm_mean - true_chirp_mass)

    inferred_cm = np.array(inferred_cm)
    cm_err = np.asarray(chirp_mass_error)

    # ------------------------------------------------------------
    # Iterative bin-size refinement
    # ------------------------------------------------------------
    edges = np.append(np.arange(CM_MIN, CM_MAX, pilot_width), CM_MAX)  # fixed pilot bins
    for it in range(n_it):
        edges = edges_from_pilot(inferred_cm, cm_err, edges, CM_MIN, CM_MAX, perr,
                                 step=step, min_events=min_events, default_size=pilot_width)
        print(f"Iteration {it + 1}/{n_it}: {len(edges) - 1} bins")

    widths = np.diff(edges)

    # ------------------------------------------------------------
    # Bin vectors and Dingo mass ranges
    # ------------------------------------------------------------
    left_edges = edges[:-1]
    right_edges = edges[1:]
    # widths = np.diff(edges) is already defined above

    range_low = left_edges - widths     # left - width
    range_high = right_edges + widths   # right + width

    print(f"{'bin':>4} {'mass low':>15} {'mass high':>16} {'mass range':>16}")
    for i in range(len(left_edges)):
        print(f"{i:>4} {range_low[i]:>15.2f} {range_high[i]:>16.2f} {range_high[i] - range_low[i]:>16.2f}")
    
    perr_per_bin, counts = binned_percentile(inferred_cm, cm_err, edges, perr)
    filled = counts > 0

    # ------------------------------------------------------------
    # Print bin edges
    # ------------------------------------------------------------
    print("Total number of bins:", len(edges) - 1)
    print("Bin edges:", np.round(edges, 2))
    print(f"{'bin':>4} {'left':>10} {'right':>10} {'width':>10} {'n':>6} {f'p{perr} |err|':>10}")
    for i in range(len(edges) - 1):
        print(f"{i:>4} {edges[i]:>10.2f} {edges[i+1]:>10.2f} {widths[i]:>10.2f} "
              f"{counts[i]:>6} {perr_per_bin[i]:>10.2f}")

    # ------------------------------------------------------------
    # Plot: one color per bin size
    # ------------------------------------------------------------
    plt.figure(figsize=(16, 10))

    # nominal bin size (a truncated last bin is grouped with its nominal size)
    nominal = step * np.ceil(widths / step - 1e-9)
    cmap = plt.get_cmap("tab10")

    #for j, size in enumerate(np.unique(nominal[filled])):
    #    sel = filled & (nominal == size)
    #    plt.bar(edges[:-1][sel], perr_per_bin[sel], width=widths[sel],
    #            align="edge", color=cmap(j % 10), edgecolor="black",
    #            alpha=0.85, label=f"bin size {size:g}")

    plt.scatter(inferred_cm, np.abs(cm_err), c=snr_events, cmap=cmap, alpha=0.5, s=10, rasterized=True, label="events")

    plt.axhline(50, linestyle="--", color="black", label="error = 50")

    for l, wi, h, n in zip(edges[:-1][filled], widths[filled],
                           perr_per_bin[filled], counts[filled]):
        plt.text(l + wi / 2, h, f"n={n}", ha="center", va="bottom", fontsize=8)

    plt.xlabel("Inferred chirp mass")
    plt.ylabel(f"{perr}th percentile of |inferred − true| chirp mass")
    plt.title(f"Chirp-mass inference error (bin size from pilot p{perr}, steps of {step})")
    plt.grid(True, axis="y")
    plt.legend(title="Bin size", loc="upper left")
    plt.tight_layout()
    plt.colorbar(label="Network SNR")
    plt.show()

    return edges, left_edges, right_edges, widths, range_low, range_high

In [ ]:
edges, left, right, widths, mr_low, mr_high = plot_chirpmass_SNR(name, 50, 1000, 90, n_it=2)

## PP-plot

In [ ]:
pp_values_chirp_mass = []
pp_values_mass_ratio = []

result_files = sorted(os.listdir(f"/scratch/tpausch/test_data/{name}/results"))

for fname in result_files:

    with open(f"/scratch/tpausch/test_data/{name}/results/{fname}", "rb") as f:

        payload = pickle.load(f)

        theta_true = payload["theta_true"]
        result = payload["result"]

        posterior_samples = result.samples


        # ============================================================
        # Chirp mass PP value
        # ============================================================

        cm_samples = posterior_samples["chirp_mass"]

        pp_cm = np.mean(
            cm_samples < theta_true["chirp_mass"]
        )

        pp_values_chirp_mass.append(pp_cm)


        # ============================================================
        # Mass ratio PP value
        # ============================================================

        #q_samples = posterior_samples["mass_ratio"]
#
        #pp_q = np.mean(
        #    q_samples < theta_true["mass_ratio"]
        #)
#
        #pp_values_mass_ratio.append(pp_q)




def plot_pp(pp_values, label):

    pp_values = np.array(pp_values)

    sorted_pp = np.sort(pp_values)

    n = len(sorted_pp)

    empirical_cdf = np.arange(1, n + 1) / n

    x = np.linspace(0, 1, 200)

    sigma = np.sqrt(x * (1 - x) / n)

    plt.figure(figsize=(6, 6))

    # Ideal calibration
    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        label="Ideal calibration",
    )


    # Empirical PP curve
    plt.plot(
        sorted_pp,
        empirical_cdf,
        marker="o",
        label=label,
    )

    plt.xlabel("Credible level")
    plt.ylabel("Fraction of injections")

    plt.title(f"PP Plot: {label}")

    plt.xlim(0, 1)
    plt.ylim(0, 1)

    plt.grid(True)
    plt.legend()

    plt.tight_layout()
    plt.show()


plot_pp(
    pp_values_chirp_mass,
    "chirp_mass",
)

#plot_pp(
#    pp_values_mass_ratio,
#    "mass_ratio",
#)

## SNR analysis

In [ ]:
snrs = []
ci90_widths = []
chirp_masses = []

result_files = sorted(os.listdir(f"/scratch/tpausch/test_data/{name}/results"))

for fname in result_files:

    with open(f"/scratch/tpausch/test_data/{name}/results/{fname}", "rb") as f:

        payload = pickle.load(f)
    
    theta_true = payload["theta_true"]
    result = payload["result"]


    snr = payload["network_snr"]   # use "ET" instead if you want the single-detector value
   

    snrs.append(snr)

    samples = result.samples["chirp_mass"].to_numpy()  # adjust if result.samples is a DataFrame

    ci90 = np.percentile(samples, 95) - np.percentile(samples, 5)
    ci90_widths.append(ci90)
    chirp_masses.append(theta_true["chirp_mass"])

snrs = np.array(snrs)
ci90_widths = np.array(ci90_widths)
chirp_masses = np.array(chirp_masses)

frac_widths = ci90_widths / chirp_masses  # relative width, dimensionless

plt.figure(figsize=(6, 5))
sc = plt.scatter(snrs, ci90_widths, c=chirp_masses, cmap="viridis", alpha=0.6, s=15)
#sc = plt.scatter(snrs, frac_widths, c=chirp_masses, cmap="viridis", alpha=0.6, s=15)
plt.xscale("log")
plt.yscale("log")
plt.xlabel("Network SNR")
plt.ylabel("Chirp mass 90% CI width")
plt.colorbar(sc, label="Injected chirp mass [M$_\\odot$]")
plt.title("posterior width vs SNR")
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()


chirp_mass_error = np.array(chirp_mass_error)
mask = snrs < 10000.0

plt.figure(figsize=(10, 5))
sc = plt.scatter(
    snrs[mask],
    chirp_mass_error[mask],
    s=10,
    edgecolor="black",
    alpha=0.3
)

plt.axhline(0, linestyle="--", color="black")
plt.xlabel("SNR")
plt.ylabel("Chirp mass inference error (inferred − true)")
plt.title("Chirp-mass inference error")
plt.grid(True)
plt.tight_layout()
plt.show()




# --- Plot distribution ---
plt.figure(figsize=(6, 5))

# Histogram, normalized to a density so it represents "how likely each value is"
counts, bins, _ = plt.hist(
    snrs, bins=100, density=True, alpha=0.5, color="steelblue", edgecolor="k", label="Histogram"
)
plt.xlabel("Network SNR")
plt.ylabel("Probability density")
plt.title(f"Distribution of injected network SNR (n={len(snrs)})")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


print(f"SNR stats: min={snrs.min():.2f}, max={snrs.max():.2f}, "
      f"median={np.median(snrs):.2f}, mean={snrs.mean():.2f}",
      f"snr99={np.percentile(np.abs(snrs), 99):.2f}")